## imports

In [25]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedPoisson

from xgboost import XGBRegressor
from catboost import CatBoostRegressor


## metrics

In [2]:
def regression_metrics(y_true, y_pred):
   return {
       "MAE": mean_absolute_error(y_true, y_pred),
       "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
       "R2": r2_score(y_true, y_pred)
   }

def evaluate(name, y_true, y_pred):
   print(f"\n{name}")
   print(regression_metrics(y_true, y_pred))


## data prep

In [3]:
df = pd.read_csv("df_reg_cleaned.csv")

y = df["injury_count"]
y_log = np.log1p(y)

# RAW (CatBoost)
X_raw = df.drop(columns=["injury_count"])

# ONE-HOT (most models)
X_ohe = pd.get_dummies(X_raw, drop_first=True)

# LABEL ENCODE (VIF branch only)
df_le = df.copy()
le = LabelEncoder()
df_le["time_bucket_encoded"] = le.fit_transform(df_le["time_bucket"])
df_le = df_le.drop(columns=["time_bucket"])

X_le = df_le.drop(columns=["injury_count"])
X_le = X_le.apply(pd.to_numeric, errors="coerce").astype(float)


## Train/Test splits

In [4]:
# OHE split
X_train_ohe, X_test_ohe, y_train, y_test = train_test_split(
   X_ohe, y, test_size=0.2, random_state=42
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Label-encoded split (VIF only)
X_train_le, X_test_le, _, _ = train_test_split(
   X_le, y, test_size=0.2, random_state=42
)

# Raw split (CatBoost)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
   X_raw, y, test_size=0.2, random_state=42
)


### vif branch

In [5]:
def compute_vif(X):
   vif = pd.DataFrame()
   vif["feature"] = X.columns
   vif["VIF"] = [
       variance_inflation_factor(X.values, i)
       for i in range(X.shape[1])
   ]
   return vif.sort_values(by="VIF", ascending=False)

print(compute_vif(X_train_le))

cols_to_drop = [
   'latitude', 'longitude',
   'time_bucket_midday',
   'time_bucket_morning_rush',
   'time_bucket_non_rush',
   'private_dr_fl'
]

X_train_vif = X_train_le.drop(columns=cols_to_drop, errors="ignore")
X_test_vif = X_test_le.drop(columns=cols_to_drop, errors="ignore")

print(compute_vif(X_train_vif))


c:\Users\aryap\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


                                   feature            VIF
3                                longitude  106301.710908
2                                 latitude  106285.819843
14                               num_units      19.818813
0                        crash_speed_limit      12.331840
21                                    hour       6.464291
19                       has_large_vehicle       3.953775
22                             day_of_week       3.393193
23                     time_bucket_encoded       3.155141
4                                 onsys_fl       2.654292
12   collision_type_grouped_single_vehicle       2.607468
10         collision_type_grouped_rear_end       1.834834
8             collision_type_grouped_other       1.811193
20                               has_other       1.630508
7           collision_type_grouped_head_on       1.555922
11        collision_type_grouped_sideswipe       1.386475
15                          has_pedestrian       1.271336
6             

## linear + Poisson

In [6]:
# Linear (raw)
lr = LinearRegression()
lr.fit(X_train_ohe, y_train)
evaluate("Linear (Raw)", y_test, lr.predict(X_test_ohe))

# Linear (log)
lr.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(lr.predict(X_test_ohe))
evaluate("Linear (Log)", y_test, y_pred)

# Poisson
poisson = PoissonRegressor(alpha=1.0)

poisson.fit(X_train_ohe, y_train)
evaluate("Poisson (Raw)", y_test, poisson.predict(X_test_ohe))

poisson.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(poisson.predict(X_test_ohe))
evaluate("Poisson (Log)", y_test, y_pred)



Linear (Raw)
{'MAE': 0.7066925023815728, 'RMSE': np.float64(1.0031708794573395), 'R2': 0.03491357363727987}

Linear (Log)
{'MAE': 0.7005592473915667, 'RMSE': np.float64(1.0252445413289184), 'R2': -0.008025004895483301}

Poisson (Raw)
{'MAE': 0.7394335088545477, 'RMSE': np.float64(1.0175007793973432), 'R2': 0.007144890676761273}

Poisson (Log)
{'MAE': 0.7382986834294648, 'RMSE': np.float64(1.040058916237769), 'R2': -0.03736659036719692}


## tree models

In [7]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_ohe, y_train)
evaluate("Random Forest", y_test, rf.predict(X_test_ohe))

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_ohe, y_train)
evaluate("Gradient Boosting", y_test, gb.predict(X_test_ohe))



Random Forest
{'MAE': 0.7260740562983666, 'RMSE': np.float64(1.0285243276411455), 'R2': -0.014484721859150618}

Gradient Boosting
{'MAE': 0.7029105760642818, 'RMSE': np.float64(0.9992673068523178), 'R2': 0.0424097147611171}


## xgboost

In [8]:
# Base
xgb = XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0)
xgb.fit(X_train_ohe, y_train)
evaluate("XGBoost (Raw)", y_test, xgb.predict(X_test_ohe))

# Log version
xgb.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(xgb.predict(X_test_ohe))
evaluate("XGBoost (Log)", y_test, y_pred)



XGBoost (Raw)
{'MAE': 0.6992560029029846, 'RMSE': np.float64(1.001109342374755), 'R2': 0.03887617588043213}

XGBoost (Log)
{'MAE': 0.6881991028785706, 'RMSE': np.float64(1.0186551594030147), 'R2': 0.004890859127044678}


## xgboost tuned

In [9]:
param_grid = {
   "n_estimators": [100, 150],
   "max_depth": [3, 4],
   "learning_rate": [0.05, 0.1],
   "subsample": [0.8],
   "colsample_bytree": [0.8],
   "gamma": [0]
}

grid = GridSearchCV(
   estimator=XGBRegressor(objective="reg:squarederror", random_state=42),
   param_grid=param_grid,
   scoring=make_scorer(r2_score),
   cv=5,
   n_jobs=-1
)

grid.fit(X_train_ohe, y_train)

best_xgb = grid.best_estimator_
print("Best params:", grid.best_params_)

evaluate("XGBoost Tuned", y_test, best_xgb.predict(X_test_ohe))


Best params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 150, 'subsample': 0.8}

XGBoost Tuned
{'MAE': 0.6982550621032715, 'RMSE': np.float64(0.9969615073621776), 'R2': 0.046823978424072266}


## zip

In [17]:
X_train_sm = sm.add_constant(X_train_vif)
X_test_sm = sm.add_constant(X_test_vif)

zip_model = ZeroInflatedPoisson(
   endog=y_train,
   exog=X_train_sm,
   exog_infl=X_train_sm,
   inflation="logit"
)

zip_res = zip_model.fit(maxiter=500, method="bfgs")

y_pred = zip_res.predict(exog=X_test_sm, exog_infl=X_test_sm)
evaluate("ZIP Model (VIF Features)", y_test, y_pred)


Optimization terminated successfully.
         Current function value: 1.136098
         Iterations: 280
         Function evaluations: 287
         Gradient evaluations: 287

ZIP Model (VIF Features)
{'MAE': 0.7132226818109346, 'RMSE': np.float64(1.00543455224701), 'R2': 0.03055319044226157}


c:\Users\aryap\anaconda3\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


## catboost

In [18]:
X_cat = X_train_raw.copy()
X_cat_test = X_test_raw.copy()

cat_cols = X_cat.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in cat_cols:
   X_cat[col] = X_cat[col].astype(str)
   X_cat_test[col] = X_cat_test[col].astype(str)

cat_features = [X_cat.columns.get_loc(col) for col in cat_cols]

cat_model = CatBoostRegressor(
   loss_function="RMSE",
   verbose=0,
   random_seed=42
)

cat_model.fit(X_cat, y_train_raw, cat_features=cat_features)

evaluate("CatBoost", y_test_raw, cat_model.predict(X_cat_test))



CatBoost
{'MAE': 0.6971245397908246, 'RMSE': np.float64(0.9976326561653974), 'R2': 0.04554009897052802}


## Model COmparisons

In [19]:
comparison_results = []

def add_result(name, y_true, y_pred, model=None, X=None):
    metrics = regression_metrics(y_true, y_pred)
    
    result = {
        "Model": name,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "AIC": None,
        "BIC": None
    }
    
    # AIC/BIC only for statistical models with likelihood
    if model is not None and X is not None:
        try:
            n = len(y_true)
            residuals = y_true - y_pred
            sse = np.sum(residuals**2)
            k = X.shape[1]

            # Gaussian likelihood assumption
            aic = n * np.log(sse/n) + 2 * k
            bic = n * np.log(sse/n) + k * np.log(n)

            result["AIC"] = aic
            result["BIC"] = bic
        except:
            pass

    comparison_results.append(result)

In [20]:
# recompute and result storage
# Linear (Raw)
add_result("Linear (Raw)", y_test, lr.predict(X_test_ohe), lr, X_test_ohe)

# Linear (Log)
lr.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(lr.predict(X_test_ohe))
add_result("Linear (Log)", y_test, y_pred, lr, X_test_ohe)

# Poisson (Raw)
poisson.fit(X_train_ohe, y_train)
add_result("Poisson (Raw)", y_test, poisson.predict(X_test_ohe), poisson, X_test_ohe)

# Poisson (Log)
poisson.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(poisson.predict(X_test_ohe))
add_result("Poisson (Log)", y_test, y_pred, poisson, X_test_ohe)

# Random Forest
add_result("Random Forest", y_test, rf.predict(X_test_ohe))

# Gradient Boosting
add_result("Gradient Boosting", y_test, gb.predict(X_test_ohe))

# XGBoost (Raw)
add_result("XGBoost (Raw)", y_test, xgb.predict(X_test_ohe))

# XGBoost (Log)
xgb.fit(X_train_ohe, y_train_log)
y_pred = np.expm1(xgb.predict(X_test_ohe))
add_result("XGBoost (Log)", y_test, y_pred)

# Tuned XGBoost
add_result("XGBoost Tuned", y_test, best_xgb.predict(X_test_ohe))

In [ ]:
# ZIP Model (uses true likelihood → take built-in AIC/BIC)
zip_aic = zip_res.aic
zip_bic = zip_res.bic
y_pred = zip_res.predict(exog=X_test_sm, exog_infl=X_test_sm)

metrics = regression_metrics(y_test, y_pred)
comparison_results.append({
    "Model": "ZIP Model (VIF Features)",
    "MAE": metrics["MAE"],
    "RMSE": metrics["RMSE"],
    "R2": metrics["R2"],
    "AIC": zip_aic,
    "BIC": zip_bic
})

# CatBoost (no AIC/BIC)
add_result("CatBoost", y_test_raw, cat_model.predict(X_cat_test))


# Convert to DataFrame and display
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(by="R2", ascending=False)

In [26]:
comparison_df

,Model,MAE,RMSE,R2,AIC,BIC
8,XGBoost Tuned,0.698255,0.996962,0.046824,NaN,NaN
10,CatBoost,0.697125,0.997633,0.045540,NaN,NaN
5,Gradient Boosting,0.702911,0.999267,0.042410,NaN,NaN
9,ZIP Model (VIF Features),0.713223,1.005435,0.030553,382882.495800,383324.014109
2,Poisson (Raw),0.739434,1.017501,0.007145,1513.444423,1738.298415
7,XGBoost (Log),0.688199,1.018655,0.004891,NaN,NaN
1,Linear (Log),0.700559,1.025245,-0.008025,2152.101282,2376.955274
4,Random Forest,0.726074,1.028524,-0.014485,NaN,NaN
3,Poisson (Log),0.738299,1.040059,-0.037367,3360.567847,3585.421839
6,XGBoost (Raw),0.704007,1.049414,-0.056111,NaN,NaN


### downloading model for Agent

In [27]:
with open('regression_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)
print("model saved.")

model saved.
